# Capturing Alternate Abbreviation Symbols Experiment

Goal of this experiment is to see how well Claude can match my annotations of gene alias pairs in literary context

In [1]:
import ssl
import certifi
import os

os.environ["SSL_CERT_FILE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context


In [12]:
import pandas as pd
import requests
from Bio import Entrez
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()
import boto3
import json

### Import manually curated table

This is a file where I manually annotated gene alias pairs by answering the following questions:
- Does this alias symbol represent this gene in this publication?
- How does this alias symbol represent this gene in this publication?

In [3]:
#Literature Defined and Alternate Abbreviation Symbol Capture- Manually Curated Set
lit_def_and_alt_abbrev_annotation_df = pd.read_csv(
    "../input/Literature Defined and Alternate Abbreviation Symbol Capture- Manually Curated Set.csv")

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,primer,methods,PCR amplification of DNA was performed using a...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,9458019,no,-,blood antibodies,methods,Incompatibility in ABO blood group antigens wa...,NaN
123,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,18822025,no,-,α1-adrenergic receptor blocker,introduction,The AUA recommends α1-adrenergic receptor bloc...,NaN
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN
125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,-,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN


### Add the official gene name for each sample from HGNC

In [4]:
def get_gene_name(hgnc_id):
    """ Retreive official gene name from HGNC for each sample in the set.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol of each sample
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    url = f"https://rest.genenames.org/fetch/hgnc_id/{hgnc_id}"
    headers = {"Accept": "application/json"}  # Request JSON format
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        try:
            gene_name = data['response']['docs'][0]['name']
            return gene_name
        except IndexError:
            return f"No gene found for HGNC ID {hgnc_id}"
    else:
        return f"Error: {response.status_code}"

BRCA1 DNA repair associated


In [5]:
gene_cache = {}

def get_gene_name_cached(hgnc_id):
    """ Retrieve the official gene name from HGNC using a cache to avoid repeated API requests for the same HGNC ID.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    if hgnc_id not in gene_cache:
        gene_cache[hgnc_id] = get_gene_name(hgnc_id)
    return gene_cache[hgnc_id]

In [6]:
lit_def_and_alt_abbrev_annotation_df["gene_name"] = lit_def_and_alt_abbrev_annotation_df["HGNC_ID"].apply(get_gene_name_cached)
lit_def_and_alt_abbrev_annotation_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,9458019,no,-,blood antibodies,methods,Incompatibility in ABO blood group antigens wa...,NaN,alpha-1-B glycoprotein
123,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,18822025,no,-,α1-adrenergic receptor blocker,introduction,The AUA recommends α1-adrenergic receptor bloc...,NaN,alpha-1-B glycoprotein
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein
125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,-,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN,alpha-1-B glycoprotein


### Add the abstract and full text for each sample

In [7]:
Entrez.email = "anastasia.bratulin@nationwidechildrens.org"

In [ ]:
def get_abstract(pmid):
    """ Retrieve the abstract text for a PubMed article given its PMID. The abstract is fetched from the NCBI PubMed database using Entrez.
    
    If the abstract consists of multiple sections, they are concatenated
    into a single string separated by blank lines.

    :param pmid: The PubMed ID (PMID) of the article
    :return: The abstract text as a string, or None if no abstract is available
    """
    handle = Entrez.efetch(db="pubmed", id=pmid, rettype="abstract", retmode="xml")
    records = Entrez.read(handle)
    handle.close()
    
    try:
        abstract = records["PubmedArticle"][0]["MedlineCitation"]["Article"]["Abstract"]["AbstractText"]
        if isinstance(abstract, list):
            return "\n\n".join(abstract)
        return abstract
    except KeyError:
        return None

In [8]:
def get_full_text(pmid):
    """ Retrieve the full-text content of a PubMed article when available via PMC.

    This function attempts to map a PubMed ID (PMID) to a PubMed Central ID
    (PMCID). If a PMCID exists and the article allows XML access, the full text
    is fetched from PMC and returned as concatenated paragraph text.

    :param pmid: The PubMed ID (PMID) of the article
    :return: The full-text content as a string, or None if unavailable or restricted
    """
    pmid = str(pmid)  
    
    # PMCID
    try:
        handle = Entrez.elink(dbfrom="pubmed", db="pmc", id=pmid)
        records = Entrez.read(handle)
        handle.close()
        pmcid = records[0]["LinkSetDb"][0]["Link"][0]["Id"]
    except (IndexError, KeyError):
        return None  # no PMC available

    # Valid PMCID Check
    if not pmcid:
        return None

    # Fetch PMC XML
    try:
        handle = Entrez.efetch(db="pmc", id=pmcid, rettype="full", retmode="xml")
        xml = handle.read()
        handle.close()
    except Exception as e:
        print(f"Error fetching PMC XML for PMCID {pmcid}: {e}")
        return None

    # Check for publisher restriction
    if b"does not allow downloading of the full text in XML form" in xml:
        return None

    # Extract paragraphs
    soup = BeautifulSoup(xml, "xml")
    paragraphs = soup.find_all("p")
    return "\n\n".join(p.get_text() for p in paragraphs)


In [10]:
lit_def_and_alt_abbrev_annotation_df["full_pmid_text"] = (
    lit_def_and_alt_abbrev_annotation_df["PMIDs from Pubtator3"]
    .progress_apply(get_full_text)
)
lit_def_and_alt_abbrev_annotation_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,full_pmid_text
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,None
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystis sp. is a zoonotic intestinal prot...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,The authors have declared that no competing in...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,9458019,no,-,blood antibodies,methods,Incompatibility in ABO blood group antigens wa...,NaN,alpha-1-B glycoprotein,"Chonghe Xu, Siqi Xie, and Meiyi Lu contributed..."
123,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,18822025,no,-,α1-adrenergic receptor blocker,introduction,The AUA recommends α1-adrenergic receptor bloc...,NaN,alpha-1-B glycoprotein,Disclosures Dr Nickel is an investigator/consu...
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein,Rhinoviruses (RVs) and respiratory enterovirus...
125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,-,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN,alpha-1-B glycoprotein,Immunoglobulin therapy including intravenous i...


In [13]:
lit_def_and_alt_abbrev_annotation_df["abstract_pmid_text"] = (
    lit_def_and_alt_abbrev_annotation_df["PMIDs from Pubtator3"]
    .progress_apply(get_abstract)
)
lit_def_and_alt_abbrev_annotation_df

100%|██████████| 127/127 [00:47<00:00,  2.69it/s]


,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,full_pmid_text,abstract_pmid_text
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None,Some genetic syndromes causing loss of hearing...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,Mice homozygous for a defect of the tub (rd5) ...
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,None,Usher syndrome is a group of diseases with aut...
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystis sp. is a zoonotic intestinal prot...,Blastocystis sp. is a zoonotic intestinal prot...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,The authors have declared that no competing in...,Blastocystis is a unicellular eukaryote common...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,9458019,no,-,blood antibodies,methods,Incompatibility in ABO blood group antigens wa...,NaN,alpha-1-B glycoprotein,"Chonghe Xu, Siqi Xie, and Meiyi Lu contributed...",Despite great efforts to promote the donation ...
123,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,18822025,no,-,α1-adrenergic receptor blocker,introduction,The AUA recommends α1-adrenergic receptor bloc...,NaN,alpha-1-B glycoprotein,Disclosures Dr Nickel is an investigator/consu...,To evaluate the safety profile and efficacy of...
124,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,26761027,no,-,genotype of rhinovirus,introduction,"A1A, A1B, A2, A23, A25, A29, A30, A31, A44, A4...",NaN,alpha-1-B glycoprotein,Rhinoviruses (RVs) and respiratory enterovirus...,Rhinoviruses (RVs) and respiratory enterovirus...
125,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,no,-,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN,alpha-1-B glycoprotein,Immunoglobulin therapy including intravenous i...,Immunoglobulin therapy including intravenous i...


#### Not all contexts are programmatically accessible

In [53]:
num_missing_abstracts = len(
    lit_def_and_alt_abbrev_annotation_df.loc[
        lit_def_and_alt_abbrev_annotation_df["abstract_pmid_text"].isna()
    ]
)

num_missing_full_texts = len(
    lit_def_and_alt_abbrev_annotation_df.loc[
        lit_def_and_alt_abbrev_annotation_df["full_pmid_text"].isna()
    ]
)

print(f"Number of abstracts that are not accessible: {num_missing_abstracts}")
print(f"Number of full texts that are not accessible: {num_missing_full_texts}")

Number of abstracts that are not accessible: 3
Number of full texts that are not accessible: 14


### Using details from each sample, generate a custom prompt for each sample

In [14]:
def generate_alt_abbrev_prompt(alias_symbol, hgnc_id, primary_gene_symbol, name):
    """ Generate an LLM prompt for curating alias gene symbols and determining their relationship to an official HGNC gene record.

    The generated prompt instructs the model to:
    - Validate whether an alias symbol corresponds to the provided HGNC record ID
    - Classify the relationship as an Alias, Alternate Abbreviation, or Other
    - Extract exact textual evidence from a biomedical context
    - Return the result as a strictly formatted JSON object

    :param alias_symbol: The alias gene symbol to be evaluated
    :param hgnc_id: The HGNC record ID associated with the primary gene
    :param primary_gene_symbol: The official HGNC gene symbol
    :param name: The official HGNC gene name
    :return: A formatted prompt string for use with a language model
    """
    prompt = f"""You are an expert biomedical scientist and gene nomenclature curator trained to identify the true gene identity of alias gene symbols and to determine the relationship that an alias symbol has with the gene it represents.
                You will be given:
                •	An alias gene symbol: {alias_symbol}
                •	A corresponding HGNC record ID: {hgnc_id}
                •	The primary (official) gene symbol: {primary_gene_symbol}
                •	The official HGNC gene name: {name}
                •	A biomedical text context in which the alias symbol appears

                Your task is to determine whether the provided HGNC record ID correctly corresponds to the gene represented by the alias symbol in the given context in a slow and careful manner.
                You must reason through the following steps internally, but you must NOT reveal your reasoning:

                Step 1: Validate Gene Identity
                •	If the HGNC record ID is incorrect, stop and indicate that the alias symbol does not represent the provided primary gene.
                •	If the HGNC record ID is correct, proceed to Step 2.

                Step 2: Assign Relationship Type
                •   If the HGNC record ID is correct, you must determine the relationship between the alias symbol and the primary gene using only one of the defined relationship types below.
                    Defined Relationship Types
                    1.	Alias
                    Use this relationship only if the biomedical text explicitly lists the alias symbol as an alternate representation alongside the official HGNC gene symbol.
                    2.	Alternate Abbreviation
                    Use this relationship if the biomedical text provides an expansion of the alias symbol that matches the official HGNC gene name.
                    3.  Other
                    Use this relationship if the biomedical text provides evidence that the alias represents the gene but not as an Alias or Alternate Abbreviation

                Step 3: Extract Evidence
                    •	The evidence must be an exact sentence from the provided biomedical text.

                Important Curation Rules
                    Do not infer an Alias relationship unless the official HGNC gene symbol appears explicitly in the text.
                    Do not force a relationship type if the evidence does not support it.
               FINAL OUTPUT REQUIREMENTS (STRICT)
                    Output ONLY a single JSON object.
                    Do NOT include step-by-step reasoning, explanations, headings, or prose.
                    Do NOT include “Step 1”, “Step 2”, or analysis text.
                    Do NOT wrap the JSON in markdown.
                    Output must begin with a left curly brace and end with a right curly brace.
                    If the HGNC record ID is incorrect, return a JSON object with:
                    "alias_symbol_represents_primary_gene": "NO"
                    "relationship_type": ""
                    "evidence": "".
                    If the HGNC record ID is correct, populate all fields appropriately.

                Output Format
                    For each alias symbol, return a JSON object using the following schema:
                    {{
                    "pmid": "EXACT PMID FROM CONTEXT (leave blank if not available)",
                    "alias_symbol": "ALIAS SYMBOL",
                    "primary_gene_symbol": "PRIMARY GENE SYMBOL",
                    "name": "OFFICIAL HGNC GENE NAME",
                    "alias_symbol_represents_primary_gene": "YES" or "NO",
                    "relationship_type": "ALIAS" or "ALTERNATE ABBREVIATION" or "OTHER" ,
                    "evidence": "EXACT SENTENCE FROM TEXT"
                    }}


             """

    return prompt

#### Prompt with just the abstract as context

In [15]:
lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df = lit_def_and_alt_abbrev_annotation_df.copy()
lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df['context'] = None
lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_prompt'] = None

In [16]:
for idx, row in lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.iterrows():
    alias_symbol = row['gene_symbol']
    primary_gene_symbol = row['primary_gene_symbol']
    name = row['gene_name']
    hgnc_id = row['HGNC_ID']
    prompt_base = generate_alt_abbrev_prompt(alias_symbol, hgnc_id, primary_gene_symbol, name)

    context = f'PMID: {row['PMIDs from Pubtator3']}\n Abstract: {row['abstract_pmid_text']}'

    full_prompt = f'{prompt_base}\n\n{context}'

    lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.at[idx,'context'] = context
    lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.at[idx,'alt_abbrev_prompt'] = full_prompt    



lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.head()

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,full_pmid_text,abstract_pmid_text,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None,Some genetic syndromes causing loss of hearing...,PMID: 9477305\n Abstract: Some genetic syndrom...,You are an expert biomedical scientist and gen...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\n Abstract: Mice homozygous for ...,You are an expert biomedical scientist and gen...
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,None,Usher syndrome is a group of diseases with aut...,PMID: 7479945\n Abstract: Usher syndrome is a ...,You are an expert biomedical scientist and gen...
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystis sp. is a zoonotic intestinal prot...,Blastocystis sp. is a zoonotic intestinal prot...,PMID: 39961042\n Abstract: Blastocystis sp. is...,You are an expert biomedical scientist and gen...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,The authors have declared that no competing in...,Blastocystis is a unicellular eukaryote common...,PMID: 38980911\n Abstract: Blastocystis is a u...,You are an expert biomedical scientist and gen...


#### Prompt with full text as context

In [17]:
lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df = lit_def_and_alt_abbrev_annotation_df.copy()
lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df['context'] = None
lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_prompt'] = None

In [18]:
for idx, row in lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.iterrows():
    alias_symbol = row['gene_symbol']
    primary_gene_symbol = row['primary_gene_symbol']
    name = row['gene_name']
    prompt_base = generate_alt_abbrev_prompt(alias_symbol, hgnc_id, primary_gene_symbol, name)

    context = f'PMID: {row['PMIDs from Pubtator3']}\n Full Text: {row['full_pmid_text']}'

    full_prompt = f'{prompt_base}\n\n{context}'

    lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.at[idx,'context'] = context
    lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.at[idx,'alt_abbrev_prompt'] = full_prompt    



lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.head()

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,full_pmid_text,abstract_pmid_text,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None,Some genetic syndromes causing loss of hearing...,PMID: 9477305\n Full Text: None,You are an expert biomedical scientist and gen...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\n Full Text: These authors contr...,You are an expert biomedical scientist and gen...
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,mouse phenotype biomarker,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,None,Usher syndrome is a group of diseases with aut...,PMID: 7479945\n Full Text: None,You are an expert biomedical scientist and gen...
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,no,-,primer,methods,The primers used in this study were: RD5: 5′–A...,NaN,TUB bipartite transcription factor,Blastocystis sp. is a zoonotic intestinal prot...,Blastocystis sp. is a zoonotic intestinal prot...,PMID: 39961042\n Full Text: Blastocystis sp. i...,You are an expert biomedical scientist and gen...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38980911,no,-,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor,The authors have declared that no competing in...,Blastocystis is a unicellular eukaryote common...,PMID: 38980911\n Full Text: The authors have d...,You are an expert biomedical scientist and gen...


In [33]:
print(lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_prompt'][0])

You are an expert biomedical scientist and gene nomenclature curator trained to identify the true gene identity of alias gene symbols and to determine the relationship that an alias symbol has with the gene it represents.
                You will be given:
                •	An alias gene symbol: rd5
                •	A corresponding HGNC record ID: HGNC:12406
                •	The primary (official) gene symbol: TUB
                •	The official HGNC gene name: TUB bipartite transcription factor
                •	A biomedical text context in which the alias symbol appears

                Your task is to determine whether the provided HGNC record ID correctly corresponds to the gene represented by the alias symbol in the given context in a slow and careful manner.
                You must reason through the following steps internally, but you must NOT reveal your reasoning:

                Step 1: Validate Gene Identity
                •	If the HGNC record ID is incorrect, stop and i

#### Test subset with abstracts as context

In [34]:
test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df = lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.iloc[:2]
test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,full_pmid_text,abstract_pmid_text,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None,Some genetic syndromes causing loss of hearing...,PMID: 9477305\n Abstract: Some genetic syndrom...,You are an expert biomedical scientist and gen...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\n Abstract: Mice homozygous for ...,You are an expert biomedical scientist and gen...


login via aws sso login in terminal

In [36]:
# Initialize the Bedrock Runtime client
bedrock = boto3.client("bedrock-runtime", region_name="us-east-2")

# Replace with your actual inference profile ID or ARN
INFERENCE_PROFILE_ID = "us.anthropic.claude-3-5-sonnet-20240620-v1:0"


def query_claude_sonnet(prompt: str) -> str:
    """ Send a prompt to the Claude Sonnet model via Amazon Bedrock and return the generated response text.

    :param prompt: The input prompt to send to the Claude Sonnet model
    :return: The model-generated response text, or an error message string
    """

    body = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1024,
        "temperature": 0.0,
        "anthropic_version": "bedrock-2023-05-31"
    }

    try:
        response = bedrock.invoke_model(
            body=json.dumps(body),
            modelId=INFERENCE_PROFILE_ID,
            contentType="application/json",
            accept="application/json"
        )
        response_body = json.loads(response["body"].read())
        return response_body["content"][0]["text"]
    except Exception as e:
        return f"[Error] {str(e)}"

about 8s ea

In [37]:
# test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'] = None

# for idx, row in test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.iterrows():
#     # Skip rows where there is no available abstract
#     if pd.isna(row['abstract_pmid_text']):
#         continue

#     test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

0 Done
1 Done


In [ ]:
# test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.to_excel('test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.xlsx')

In [ ]:
test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df = pd.read_excel("test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.xlsx")

In [38]:
print(test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'][0])

{
    "pmid": "9477305",
    "alias_symbol": "rd5",
    "primary_gene_symbol": "TUB",
    "name": "TUB bipartite transcription factor",
    "alias_symbol_represents_primary_gene": "YES",
    "relationship_type": "ALIAS",
    "evidence": "Mice homozygous for the tub (rd5) mutation exhibit progressive retinal degeneration, sensorineural hearing loss, reduced fertility, and obesity, and presently represent the only animal model with neuroepithelial degeneration of both cochlea and retina without other neurological abnormalities."
}


In [39]:
print(test_lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'][1])

{
    "pmid": "9390831",
    "alias_symbol": "rd5",
    "primary_gene_symbol": "TUB",
    "name": "TUB bipartite transcription factor",
    "alias_symbol_represents_primary_gene": "YES",
    "relationship_type": "ALIAS",
    "evidence": "Mice homozygous for a defect of the tub (rd5) gene exhibit cochlear and retinal degeneration combined with obesity, and resemble certain human autosomal recessive sensory deficit syndromes."
}


In [46]:
len(lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df)

127

#### Whole set with abstracts as context

In [ ]:
# lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'] = None

# for idx, row in lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.iterrows():
#     # Skip rows where there is no available abstract
#     if pd.isna(row['abstract_pmid_text']):
#         continue

#     lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

In [65]:
# lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.to_excel('lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.xlsx')

In [ ]:
lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df = pd.read_excel("lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df.xlsx")

In [66]:
print(lit_def_and_alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_response'][126])

{
    "pmid": "2447112",
    "alias_symbol": "A1B",
    "primary_gene_symbol": "A1BG",
    "name": "alpha-1-B glycoprotein",
    "alias_symbol_represents_primary_gene": "YES",
    "relationship_type": "ALTERNATE ABBREVIATION",
    "evidence": "Alpha 1-beta glycoprotein (A1B) was purified to homogeneity from human plasma using a three-step procedure involving pseudo-ligand affinity chromatography on immobilized Cibacron Blue 3-GA, gel filtration chromatography on Sephadex G-200 and ion-exchange chromatography on DEAE Affigel Blue."
}


#### Test subset with full text as context

In [40]:
test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df = lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.iloc[:2]
test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,full_pmid_text,abstract_pmid_text,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None,Some genetic syndromes causing loss of hearing...,PMID: 9477305\n Full Text: None,You are an expert biomedical scientist and gen...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\n Full Text: These authors contr...,You are an expert biomedical scientist and gen...


In [42]:
# test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'] = None

# for idx, row in test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.iterrows():
#     # Skip rows where there is no available full text
#     if pd.isna(row['full_pmid_text']):
#         continue

#     test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

1 Done


In [ ]:
# test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.to_excel('test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx')

In [ ]:
test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df = pd.read_excel("test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx")

In [45]:
print(test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'][0])

None


In [46]:
print(test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'][1])

Here is the JSON output based on the provided information:

{
"pmid": "9390831",
"alias_symbol": "rd5",
"primary_gene_symbol": "TUB",
"name": "TUB bipartite transcription factor",
"alias_symbol_represents_primary_gene": "YES",
"relationship_type": "OTHER",
"evidence": "A truncating homozygous variant in this gene had previously been reported in a single family with three subjects sharing retinal dystrophy and obesity."
}


In [44]:
test_lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,How?,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,full_pmid_text,abstract_pmid_text,context,alt_abbrev_prompt,alt_abbrev_response
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,mouse phenotype biomarker,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None,Some genetic syndromes causing loss of hearing...,PMID: 9477305\n Full Text: None,You are an expert biomedical scientist and gen...,None
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,mouse phenotype biomarker,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\n Full Text: These authors contr...,You are an expert biomedical scientist and gen...,Here is the JSON output based on the provided ...


#### Whole set with full text as context

In [49]:
# lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'] = None

# for idx, row in lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.iterrows():
#     # Skip rows where there is no available full text
#     if pd.isna(row['full_pmid_text']):
#         continue

#     lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

1 Done
3 Done
4 Done
5 Done
6 Done
7 Done
8 Done
9 Done
11 Done
12 Done
14 Done
18 Done
19 Done
20 Done
21 Done
22 Done
24 Done
25 Done
26 Done
27 Done
28 Done
29 Done
30 Done
31 Done
32 Done
33 Done
34 Done
35 Done
36 Done
37 Done
38 Done
39 Done
41 Done
42 Done
43 Done
44 Done
45 Done
46 Done
47 Done
48 Done
49 Done
50 Done
51 Done
52 Done
53 Done
54 Done
55 Done
56 Done
57 Done
59 Done
60 Done
61 Done
62 Done
63 Done
64 Done
65 Done
66 Done
68 Done
69 Done
70 Done
71 Done
72 Done
73 Done
74 Done
75 Done
76 Done
77 Done
78 Done
79 Done
80 Done
81 Done
82 Done
83 Done
84 Done
85 Done
86 Done
87 Done
88 Done
89 Done
90 Done
92 Done
93 Done
94 Done
95 Done
96 Done
97 Done
98 Done
99 Done
100 Done
101 Done
102 Done
103 Done
104 Done
105 Done
106 Done
107 Done
108 Done
109 Done
110 Done
111 Done
112 Done
113 Done
115 Done
116 Done
117 Done
118 Done
119 Done
120 Done
121 Done
122 Done
123 Done
124 Done
125 Done


In [50]:
# lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.to_excel('lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx')

In [ ]:
lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df = pd.read_excel("lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df.xlsx")

In [52]:
#output should be none since there is no full text accessible programmatically for this sample
print(lit_def_and_alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_response'][126])

None
